In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_account_move AS
SELECT
    move_id,
    name,
    payment_reference,
    move_type,
    state,
    ref,
    created_at,
    updated_at,
    is_updated,
    is_deleted
FROM bakery_pos_develop.silver.account_move
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY move_id
    ORDER BY updated_at DESC
) = 1

    

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.gold.dim_account_move (
    move_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    move_id BIGINT,
    name STRING,
    payment_reference STRING,
    move_type STRING,
    state STRING,
    ref STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN,
    PRIMARY KEY (move_sk)
)

In [0]:
%sql
MERGE INTO bakery_pos_develop.gold.dim_account_move AS tgt
USING stg_account_move AS src
ON tgt.move_id = src.move_id

WHEN MATCHED AND src.updated_at > tgt.updated_at THEN
    UPDATE SET
        tgt.name = src.name,
        tgt.payment_reference = src.payment_reference,
        tgt.move_type = src.move_type,
        tgt.state = src.state,
        tgt.ref = src.ref,
        tgt.created_at = src.created_at,
        tgt.updated_at = src.updated_at,
        tgt.is_updated = src.is_updated,
        tgt.is_deleted = src.is_deleted

WHEN NOT MATCHED THEN
    INSERT (
        move_id,
        name,
        payment_reference,
        move_type,
        state,
        ref,
        created_at,
        updated_at,
        is_updated,
        is_deleted
    )
    VALUES (
        src.move_id,
        src.name,
        src.payment_reference,
        src.move_type,
        src.state,
        src.ref,
        src.created_at,
        src.updated_at,
        src.is_updated,
        src.is_deleted
    )

In [0]:
%sql

SELECT * FROM bakery_pos_develop.gold.dim_account_move limit 10